1. Phân loại 3 memory từ ví dụ.
"Tôi thích bay chuyến bay buổi sáng" -> SEMANTIC (preference ổn định, phát biểu chung tôi thích)
"Tuần sau đặt giúp tôi chuyên đi Bangkok" -> WORKING (task hiện tại) + EPISODIC (log sau khi thực hiện)
"À tôi không thích transit quá 2 tiếng" -> SEMANTIC (constraint, có tín hiệu phủ định mạnh không thích)


2. Schema thiết kế

// Working memory - tồn tại trong session, có TTL, xóa khi task xong
{
    "user_id": "u_123",
    "memory_type": "working",
    "content": "Đang tìm chuyến bay Bangkok cho tuần sau (15-21/09), yêu cầu: bay sáng, transit <=2h",
    "created_at": "2026-09-08T10:00:00Z",
    "confidence": 1.0,
    "session_id": "s_456",
    "ttl_expires_at": "2026-09-08T22:00:00Z",
    "status": "in_progress"
}

// Episodic memory — ghi nhận 1 sự kiện/tương tác đã xảy ra, không tự sinh ra hành vi mới, dùng để tra cứu lịch sử
{
  "user_id": "u_123",
  "memory_type": "semantic",
  "content": "Ưu tiên chuyến bay khởi hành buổi sáng",
  "created_at": "2026-09-08T10:00:00Z",
  "confidence": 0.6,
  "category": "flight_preference",
  "evidence_count": 1,
  "last_reinforced_at": "2026-09-08T10:00:00Z",
  "polarity": "prefer"
}
{
  "user_id": "u_123",
  "memory_type": "semantic",
  "content": "Không chấp nhận transit quá 2 tiếng",
  "created_at": "2026-09-08T10:00:00Z",
  "confidence": 0.85,
  "category": "flight_constraint",
  "evidence_count": 1,
  "last_reinforced_at": "2026-09-08T10:00:00Z",
  "polarity": "reject",
  "
}

Field bổ sung ngoài schema gốc: ttl_expires_at/status (working memory phải tự dọn, không thể phình vô hạn), evidence_count/last_reinforced_at (semantic memory cần cơ chế củng cố 1 lần), polarity/is_hard_constraint (để agent biết filter cứng hay chỉ rank ưu tiên), related_task_id (link episodic về task để audit)


5 câu hỏi bắt buộc:

1. Failure mode lớn nhất: Lưu nhầm memory tạm thời thành semantic vĩnh viễn -> agent áp dụng sai cho mọi lần sau. Ngược lại cũng nguy hiểm, bỏ sót correction khiến agent gặp lại lỗi cũ. 

2. Biết hệ thống đã tốt lên khi nào?
Tỷ lệ preference được agent áp dụng đúng ở lần tương tác sau tăng lên (vd đặt vé lần 2 tự động lọc transit <=2h mà không cần user nhắc lại); số lần user phải sửa lại agent giảm dần.

3. Metric cần theo dõi: precision/recall của should_store_memory so với nhãn của người gán; tỷ lệ semantic memory bị mâu thuẫn nhau theo thòi gian; độ trễ khi truy xuất memory relevant cho 1 câu hỏi mới.

4. Có thực sự cần agent không? 
Cho use case đặt vé đơn giản có preference cố  định, một form/rule-based booking assistant có thể đủ; Agent + memory thực sự cần thiết khi input tự nhiên, đa dạng, và preference/constraint được tiết lộ rải tác qua nhiều hội thoại như trên.

5. Kiến trúc đơn giản hơn:
Thay vì 3 loại memory phức tạp ngay từ đầu, có thể bắt đầu chỉ với semantic memory dạng key-value profile (structured slots: preferred_time, max_transit_hours, diet...) cập nhật qua slot filling, và chỉ thêm episodic log khi thực sự cần audit lịch sử - giảm độ phức tạp cho giai đoạn đầu.